In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

print("Train:", train.shape)
print("Test:", test.shape)

In [ ]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

print("Train:", train.shape)
print("Test:", test.shape)

display(train.head())

train.info()

In [ ]:
train["Survived"].value_counts()

In [ ]:
train["Survived"].value_counts(normalize=True)

In [ ]:
train.groupby("Sex")["Survived"].mean()

In [ ]:
train.groupby("Pclass")["Survived"].mean()

In [ ]:
train.groupby(["Sex", "Pclass"])["Survived"].mean()

In [ ]:
import matplotlib.pyplot as plt

train.groupby(["Sex", "Pclass"])["Survived"].mean().unstack().plot(kind="bar")

plt.ylabel("Survival Rate")
plt.title("Titanic Survival Rate by Sex and Pclass")
plt.xticks(rotation=0)
plt.show()

In [ ]:
train["Age"].isna().sum()

In [ ]:
train["Age"].describe()

In [ ]:
train.groupby("Pclass")["Age"].mean()

In [ ]:
train.groupby(["Sex", "Pclass"])["Age"].mean()

In [ ]:
# 计算每个 Sex + Pclass 组合的平均年龄
age_mean = train.groupby(["Sex", "Pclass"])["Age"].transform("mean")

# 用对应组合的平均年龄填补缺失值
train["Age"] = train["Age"].fillna(age_mean)

# 检查还有多少缺失值
print("剩余缺失年龄：", train["Age"].isna().sum())

In [ ]:
train[train["Embarked"].isna()]

In [ ]:
train["Embarked"].value_counts()

In [ ]:
train["Embarked"] = train["Embarked"].fillna(
    train["Embarked"].mode()[0]
)

print("Embarked剩余缺失值：", train["Embarked"].isna().sum())

In [ ]:
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]

X = pd.get_dummies(train[features], drop_first=True)
y = train["Survived"]

print(X.head())
print("\n特征数量：", X.shape[1])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("训练集：", X_train.shape)
print("验证集：", X_valid.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

print("模型训练完成")

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_valid)

print("验证集准确率：", accuracy_score(y_valid, y_pred))

In [ ]:
pd.Series(model.coef_[0], index=X.columns).sort_values()

In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_valid, y_pred))

In [ ]:
train["Title"] = train["Name"].str.extract(r",\s*([^.]*)\.")
    
print(train["Title"].value_counts())

In [ ]:
train.groupby("Title")["Survived"].agg(["count", "mean"]).sort_values("mean", ascending=False)

In [ ]:
# 先统一一些特殊称谓
train["Title"] = train["Title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
})

# 保留主要称谓，其余归为 Rare
main_titles = ["Mr", "Miss", "Mrs", "Master"]

train.loc[~train["Title"].isin(main_titles), "Title"] = "Rare"

print(train["Title"].value_counts())

In [ ]:
# 从 test 的 Name 中提取 Title
test["Title"] = test["Name"].str.extract(r",\s*([^.]*)\.")

# 统一特殊称谓
test["Title"] = test["Title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
})

# 其余少见称谓统一为 Rare
test.loc[~test["Title"].isin(main_titles), "Title"] = "Rare"

print(test["Title"].value_counts())

In [ ]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "Title"
]

X = pd.get_dummies(train[features], drop_first=True)
y = train["Survived"]

print(X.head())
print("\n特征数量：", X.shape[1])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("训练集：", X_train.shape)
print("验证集：", X_valid.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

model_title = LogisticRegression(max_iter=1000)

model_title.fit(X_train, y_train)

print("加入 Title 后，模型训练完成")

In [ ]:
from sklearn.metrics import accuracy_score

y_pred_title = model_title.predict(X_valid)

print("加入 Title 后的验证集准确率：", accuracy_score(y_valid, y_pred_title))

In [ ]:
print("预测结果完全相同：", (y_pred == y_pred_title).all())
print("预测结果发生变化的人数：", (y_pred != y_pred_title).sum())

In [ ]:
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

print(
    train.groupby("FamilySize")["Survived"]
    .agg(["count", "mean"])
)

In [ ]:
def get_family_type(size):
    if size == 1:
        return "Alone"
    elif size <= 4:
        return "Small"
    else:
        return "Large"

train["FamilyType"] = train["FamilySize"].apply(get_family_type)
test["FamilyType"] = test["FamilySize"].apply(get_family_type)

print(
    train.groupby("FamilyType")["Survived"]
    .agg(["count", "mean"])
)

In [ ]:
# 1. 加入 FamilyType
features = [
    "Pclass", "Sex", "Age", "SibSp", "Parch",
    "Fare", "Embarked", "Title", "FamilyType"
]

X = pd.get_dummies(train[features], drop_first=True)
y = train["Survived"]

# 2. 划分训练集和验证集
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. 训练模型
model_family = LogisticRegression(max_iter=1000)
model_family.fit(X_train, y_train)

# 4. 评价
y_pred_family = model_family.predict(X_valid)

print("加入 FamilyType 后准确率：",
      accuracy_score(y_valid, y_pred_family))

In [ ]:
# 处理 Test 中的缺失值
test["Age"] = test["Age"].fillna(
    test.groupby(["Sex", "Pclass"])["Age"].transform("mean")
)

test["Age"] = test["Age"].fillna(train["Age"].median())
test["Fare"] = test["Fare"].fillna(train["Fare"].median())

# 准备测试集特征
X_test = pd.get_dummies(test[features], drop_first=True)

# 保证 train 和 test 的特征完全一致
X_test = X_test.reindex(columns=X.columns, fill_value=0)

# 预测
test_pred = model_family.predict(X_test)

# 生成提交文件
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_pred
})

submission.to_csv("submission.csv", index=False)

print(submission.head())
print("\n提交文件已生成：submission.csv")